In [11]:
import json
with open("../datos/web/pedidos_web.json", encoding="utf-8") as f:
    web = json.load(f)
print(type(web), list(web.keys()))
print(web["count"], len(web["orders"]))
print(web["orders"][0]["customer"]["name"])
print(web["orders"][0]["items"][0])

<class 'dict'> ['source', 'exported_at', 'count', 'orders']
7956 7956
Bar La Castellana
{'sku': 'AGU-0014', 'qty': 1, 'unit_price': 9.13, 'discount': 5.0}


In [13]:
import duckdb
con = duckdb.connect("../db/marejada.duckdb", read_only=True)
con.sql("SELECT count(*) AS lineas, round(sum(importe_neto)) AS ventas FROM gold.fact_ventas")

┌────────┬────────────┐
│ lineas │   ventas   │
│ int64  │   double   │
├────────┼────────────┤
│ 366928 │ 42633273.0 │
└────────┴────────────┘

In [ ]:
filas = con.sql("""
SELECT c.provincia, round(sum(f.importe_neto)) AS ventas
FROM gold.fact_ventas AS f
JOIN gold.dim_cliente AS c ON c.cliente_key = f.cliente_key
JOIN gold.dim_fecha AS d ON d.fecha_key = f.fecha_key
WHERE d.anio = 2026
GROUP BY c.provincia
ORDER BY ventas DESC
""").fetchall()
print(type(filas), len(filas))
print(filas[0])
for provincia, ventas in filas:
    print(f"{provincia:<12} {ventas:>14,.0f} €")

<class 'list'> 5
('Cantabria', 7985003.0)
Cantabria         7,985,003 €
Asturias          2,536,086 €
Bizkaia           1,635,110 €
Burgos            1,284,843 €
Palencia            742,344 €


In [16]:
total, = con.sql("SELECT sum(importe_neto) FROM gold.fact_ventas").fetchone()
print(f"{total:,.2f}")

42,633,273.34


In [17]:
con.close()

In [12]:
import marejada_utils as mu
print(mu.FECHA_CORTE)
print(mu.normalizar_sku("cer0012"))
print(mu.a_decimal("102,00") * 3)

2026-09-30
CER-0012
306.00


In [25]:
def resumen(valores):
    return min(valores), max(valores), sum(valores) / len(valores)
minimo, maximo, media = resumen([1187705, 1066618, 1351890, 1500290])
print(f"Mínimo {minimo:,} · máximo {maximo:,} · media {media:,.0f}")

Mínimo 1,066,618 · máximo 1,500,290 · media 1,276,626


In [2]:
from pathlib import Path
datos = Path('../datos')
print (datos.exists())

for fichero in sorted(datos.rglob('*.csv')):
    print(f'{fichero.stat().st_size:>12,}{fichero}')

True
      31,907../datos/almacen/inventario_2026-09-30.csv
     184,713../datos/erp/clientes.csv
  13,284,564../datos/erp/lineas_pedido.csv
   6,436,477../datos/erp/pedidos.csv
      28,095../datos/erp/productos.csv
       1,541../datos/rrhh/comerciales.csv


In [ ]:
import csv
with open("../datos/rrhh/comerciales.csv", encoding="latin-1", newline="") as f:
    comerciales = list(csv.DictReader(f))
print(len(comerciales))
print(comerciales[1])

In [3]:
ruta = Path("../datos/LEEME.md")
with open(ruta, encoding="utf-8") as f:
    lineas = f.readlines()
print(len(lineas), "líneas")
print(lineas[0])

19 líneas
# Datos de Marejada Distribución, S.L. · versión 1



In [26]:
def anadir_aviso(aviso, avisos=None):
    if avisos is None:
        avisos = []
    avisos.append(aviso)
    return avisos

print(anadir_aviso("Stock bajo"))
print(anadir_aviso("Pedido urgente"))

['Stock bajo']
['Pedido urgente']


In [27]:
def anadir_aviso(aviso, avisos=[]):
    avisos.append(aviso)
    return avisos

print(anadir_aviso("Stock bajo"))
print(anadir_aviso("Pedido urgente"))

['Stock bajo']
['Stock bajo', 'Pedido urgente']


In [18]:
pedidos = [
("2026025160", "Web B2B"), ("2026025161", "Comercial"), ("2026025162", "Teléfono"),
("2026025163", "Comercial"), ("2026025164", "Web B2B"), ("2026025165", "Comercial"),
]

por_canal = {}
for pedido_id, canal in pedidos: 
    por_canal[canal] = por_canal.get(canal, 0)+1

print (por_canal)

{'Web B2B': 2, 'Comercial': 3, 'Teléfono': 1}


In [24]:
def importe_neto(cantidad, precio, descuento_pct=0):
    return cantidad * precio * (1 - descuento_pct / 100)
print(importe_neto(6, 41.65))
print(importe_neto(6, 41.65, 10))
print(importe_neto(precio=41.65, cantidad=6, descuento_pct=10))

249.89999999999998
224.91
224.91


In [22]:
def normalizar_sku(sku):
    #Devuelve el SKU en el formato del maestro: CER-0001
    limpio = sku.strip().upper().replace("_", "-")
    return limpio[:3] + "-" + limpio[3:]

print (normalizar_sku(" cer-0012"))
print (normalizar_sku("CER0007"))
print (normalizar_sku("VIN-0025"))

CER--0012
CER-0007
VIN--0025


In [20]:
stock = 30
semanas = 0
while stock >0:
    stock -= 8 #se vende 8 unidades por semana
    semanas += 1
print (f'Se acabará en {semanas} semanas')

Se acabará en 4 semanas


In [21]:
skus = [" cer-0012", "CER0007", "VIN-0025 ", "agu-0001"]
limpios = [s.strip().upper() for s in skus]
print (limpios)

solo_cervezas = [s for s in limpios if s.startswith("CER")]
print (solo_cervezas)

['CER-0012', 'CER0007', 'VIN-0025', 'AGU-0001']
['CER-0012', 'CER0007']


In [14]:
for i in range(3):
    print('Vuelta', i)

almacenes = ['Guarnizo', 'Llanera']
for posicion, almacen in enumerate(almacenes, start=1):
    print(posicion, almacen)

meses = ['jul', 'ago', 'sep']
ventas = [2142228, 1975639, 1571643]
for mes, v in zip (meses, ventas):
    print (mes,f'{v:12,}€'.replace(',', '.'))

Vuelta 0
Vuelta 1
Vuelta 2
1 Guarnizo
2 Llanera
jul    2.142.228€
ago    1.975.639€
sep    1.571.643€


In [12]:
ventas_mes = {'2026-07': 2142228, "2026-08": 1975639, "2026-09": 1571643}
for mes, ventas in ventas_mes.items():
    print (f'{mes}: {ventas:12,}€'.replace(',', '.'))

2026-07:    2.142.228€
2026-08:    1.975.639€
2026-09:    1.571.643€


In [46]:
email = ''
if email: 
    print('enviar a', email)
else: 
    print('cliente sin email')

cliente sin email


In [45]:
email = 'cliente@example.com'
if email: 
    print('enviar a', email)
else: 
    print('cliente sin email')

enviar a cliente@example.com


In [41]:
gastos_envio = 18.0

if gastos_envio == 0:
    tipo = 'Portes pagados'
elif gastos_envio == 12:
    tipo = 'Portes Cantabria'
else:
    tipo = 'Portes fuera de Cantabria'
print (tipo)

Portes fuera de Cantabria


In [40]:
canales = ['comercial', 'telefono', 'comercial', 'web b2b', 'telefono']
print (set(canales))
print (len(set(canales)))

web_2025 = {10001, 10002, 10005}
web_2026 = {10002, 10005, 10007}
print (web_2026 - web_2025)
print (web_2025 & web_2026)

{'web b2b', 'telefono', 'comercial'}
3
{10007}
{10002, 10005}


In [36]:
cliente = {
    'cliente_id': 10001,
    'nombre': 'bar la ola',
    'provincia': 'cantabria',
    'activo': True,
}
print(cliente['nombre'])
cliente['telefono'] = '979 94 00 79'
print(cliente.keys())
print(len(cliente))
print (cliente)
print (cliente.get('email'))
print (cliente.get('email', 'sin email'))

bar la ola
dict_keys(['cliente_id', 'nombre', 'provincia', 'activo', 'telefono'])
5
{'cliente_id': 10001, 'nombre': 'bar la ola', 'provincia': 'cantabria', 'activo': True, 'telefono': '979 94 00 79'}
None
sin email


In [31]:
fila = (10001, 'bar la ola', 'cantabria')
print (fila[1])
cliente_id, nombre, provincia = fila
print (nombre, '-', provincia)

bar la ola
bar la ola - cantabria


In [30]:
provincias = ['Cantabria', 'Asturias']
copias = provincias.copy()
copias.append('Bizkaia')
print(provincias)
print(copias)

['Cantabria', 'Asturias']
['Cantabria', 'Asturias', 'Bizkaia']


In [29]:
provincias = ['Cantabria', 'Asturias']
copia = provincias
copia.append('Bizkaia')
print(provincias)

['Cantabria', 'Asturias', 'Bizkaia']


In [28]:
almacenes.append('Bilbao')
print(almacenes)
almacenes.remove('Bilbao')
print(almacenes)
print('Llanera' in almacenes)

['guarnizo', 'llanera', 'Bilbao']
['guarnizo', 'llanera']
False


In [27]:
almacenes=['guarnizo','llanera']
ventas_mes = [1187705, 1066618, 1351890, 1500290]

print(len(ventas_mes))
print(ventas_mes[0])
print(ventas_mes[-1])
print(ventas_mes[1:3])
print(sum(ventas_mes),max(ventas_mes),min(ventas_mes))

4
1187705
1500290
[1066618, 1351890]
5106503 1500290 1066618


In [26]:
precio = 41.65
print(precio > 40)
print(precio == 41.65)
print(precio != 41.65)
print(10 < precio <= 50)

email = None
print(email is None)

True
True
False
True
True


In [21]:
print(int('42') + 1)
print (float('41,65'.replace(",", ".")))
print (str(1212)+' clientes')
print(bool(''), bool('hola'), bool(0), bool(42))
print(str('cuarenta'))


43
41.65
1212 clientes
False True False True
cuarenta


In [15]:
cliente='bar la ola'
pedidos=47
ventas=18234.567

print(f'El cliente {cliente} ha hecho {pedidos} pedidos')
print(f'Ventas: {ventas:.2f} €')
print(f'Ventas: {ventas:,.2f} €')
print(f'Media per pedido: {ventas/pedidos:.1f} €')

El cliente bar la ola ha hecho 47 pedidos
Ventas: 18234.57 €
Ventas: 18,234.57 €
Media per pedido: 388.0 €


In [14]:
print("CER-0007".replace("-", ""))
print(len("CER-0007"))

CER0007
8


In [12]:
provincia = "Cantabria"
print(provincia.strip())
print(provincia.strip().title().replace("a", "A"))
print(provincia.lower())

Cantabria
CAntAbriA
cantabria


In [9]:
from decimal import Decimal

print (0.1 + 0.2)
print (Decimal ('0.1') + Decimal('0.2'))
print (Decimal("41.65") * 6)

0.30000000000000004
0.3
249.90


In [6]:
print (7/2)
print (7//2)
print (7%2)
print (2**10)
print(round(2.675, 2))

3.5
3
1
1024
2.67


In [5]:
print(type(1212))
print(type(41.65))
print(type("Bar La Ola"))
print(type(True))
print(type(None))

<class 'int'>
<class 'float'>
<class 'str'>
<class 'bool'>
<class 'NoneType'>


In [2]:
precio = 41.65
cantidad = 6
print("Importe calculado")
precio * cantidad

Importe calculado


249.89999999999998

In [1]:
print ("Hola, Marejada")

Hola, Marejada
